In [1]:
import subprocess, sys
try:
    import torch_geometric
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch-geometric", "scikit-learn", "-q"], check=False)
    import torch_geometric
print(f"torch-geometric: {torch_geometric.__version__}")


torch-geometric: 2.8.0.post1


# Temporal Data Splitting

Splits graph tensors chronologically into Train (60%), Validation (20%), and Test (20%) sets.


## 1. Load Graph Tensors


In [2]:
import os, glob
import torch

LOCAL_PATH = '/home/shreyas-nalle/Desktop/Delusional/model/graph_data.pt'
paths_to_check = [LOCAL_PATH, 'model/graph_data.pt', 'graph_data.pt']
kaggle_glob = glob.glob('/kaggle/input/**/graph_data*', recursive=True)
if kaggle_glob: paths_to_check.insert(0, kaggle_glob[0])

graph_path = next((p for p in paths_to_check if os.path.exists(p)), None)
if graph_path is None:
    raise FileNotFoundError('graph_data.pt not found!')

graph_data = torch.load(graph_path, weights_only=False)
x = graph_data['x']
edge_index = graph_data['edge_index']
edge_attr = graph_data['edge_attr']
timestamps = graph_data['timestamps']
y = graph_data['y']
print(f'Loaded graph data from {graph_path}: {x.shape[0]:,} nodes, {edge_index.shape[1]:,} edges')


Loaded graph data from graph_data.pt: 1,754,264 nodes, 5,000,000 edges


## 2. Bucket Transactions into Days


In [3]:
import numpy as np
import itertools

n_days = int(timestamps.max() / (3600 * 24) + 1)
daily_irs = []
daily_inds = []
daily_trans = []

for day in range(n_days):
    l = day * 24 * 3600
    r = (day + 1) * 24 * 3600
    day_inds = torch.where((timestamps >= l) & (timestamps < r))[0]
    daily_irs.append(y[day_inds].float().mean().item())
    daily_inds.append(day_inds)
    daily_trans.append(day_inds.shape[0])

print(f"Processed {n_days} calendar days")


Processed 69 calendar days


## 3. Find Optimal Split Cut-Points


In [4]:
split_per = [0.6, 0.2, 0.2]
daily_totals = np.array(daily_trans)
d_ts = daily_totals
I = list(range(len(d_ts)))
split_scores = {}

for i, j in itertools.combinations(I, 2):
    if j >= i:
        split_totals = [d_ts[:i].sum(), d_ts[i:j].sum(), d_ts[j:].sum()]
        split_totals_sum = np.sum(split_totals)
        split_props = [v / split_totals_sum for v in split_totals]
        split_error = [abs(v - t) / t for v, t in zip(split_props, split_per)]
        score = max(split_error)
        split_scores[(i, j)] = score

best_i, best_j = min(split_scores, key=split_scores.get)
split = [
    list(range(best_i)),
    list(range(best_i, best_j)),
    list(range(best_j, len(daily_totals)))
]
print(f"Optimal cut-points: Day {best_i} and Day {best_j}")


Optimal cut-points: Day 1 and Day 2


## 4. Collect Transaction Indices


In [5]:
split_inds = {k: [] for k in range(3)}
for k in range(3):
    for day in split[k]:
        split_inds[k].append(daily_inds[day])

tr_inds = torch.cat(split_inds[0])
val_inds = torch.cat(split_inds[1])
te_inds = torch.cat(split_inds[2])
total = y.shape[0]
print(f"Train: {tr_inds.shape[0]:,} ({tr_inds.shape[0]/total*100:.1f}%), Val: {val_inds.shape[0]:,} ({val_inds.shape[0]/total*100:.1f}%), Test: {te_inds.shape[0]:,} ({te_inds.shape[0]/total*100:.1f}%)")


Train: 4,466,821 (89.3%), Val: 530,666 (10.6%), Test: 2,513 (0.1%)


## 5. Create PyG Data Objects


In [6]:
from torch_geometric.data import Data

tr_x = val_x = te_x = x
e_tr = tr_inds.numpy()
e_val = np.concatenate([tr_inds.numpy(), val_inds.numpy()])

tr_edge_index = edge_index[:, e_tr]
tr_edge_attr = edge_attr[e_tr]
tr_y = y[e_tr]
tr_edge_times = timestamps[e_tr]

val_edge_index = edge_index[:, e_val]
val_edge_attr = edge_attr[e_val]
val_y = y[e_val]
val_edge_times = timestamps[e_val]

te_edge_index = edge_index
te_edge_attr = edge_attr
te_y = y
te_edge_times = timestamps

tr_data = Data(x=tr_x, edge_index=tr_edge_index, edge_attr=tr_edge_attr, y=tr_y)
val_data = Data(x=val_x, edge_index=val_edge_index, edge_attr=val_edge_attr, y=val_y)
te_data = Data(x=te_x, edge_index=te_edge_index, edge_attr=te_edge_attr, y=te_y)

tr_data.timestamps = tr_edge_times
val_data.timestamps = val_edge_times
te_data.timestamps = te_edge_times
print(f"Created Data objects: Train ({tr_data.num_edges:,} edges), Val ({val_data.num_edges:,} edges), Test ({te_data.num_edges:,} edges)")


Created Data objects: Train (4,466,821 edges), Val (4,997,487 edges), Test (5,000,000 edges)


## 6. Normalize Features


In [7]:
def z_norm(data):
    std = data.std(0).unsqueeze(0)
    std = torch.where(std == 0, torch.tensor(1, dtype=torch.float), std)
    return (data - data.mean(0).unsqueeze(0)) / std

tr_data.x = val_data.x = te_data.x = z_norm(tr_data.x)
tr_data.edge_attr = z_norm(tr_data.edge_attr)
val_data.edge_attr = z_norm(val_data.edge_attr)
te_data.edge_attr = z_norm(te_data.edge_attr)
print("Features normalized via Z-score")


Features normalized via Z-score


## 7. Save Split Data Objects


In [8]:
save_path = "split_data.pt"
torch.save({"tr_data": tr_data, "val_data": val_data, "te_data": te_data, "tr_inds": tr_inds, "val_inds": val_inds, "te_inds": te_inds}, save_path)
print(f"Saved split_data.pt -> {save_path} ({os.path.getsize(save_path) / 1e9:.2f} GB)")


Saved split_data.pt -> split_data.pt (0.91 GB)
